In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import os
import json

import sys

sys.path.append("../")

os.getcwd()
os.chdir("../../")
os.getcwd()

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7" #! NOTE: CHANGE TO 0,1 FOR VAST AI
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

import torch
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import transformers

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2025-07-16 17:31:12 __main__ INFO     torch.__version__='2.7.1+cu126', torch.version.cuda='12.6'
2025-07-16 17:31:13 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2025-07-16 17:31:13 __main__ INFO     transformers.__version__='4.53.0'


In [6]:
from src.utils.training_utils import get_device_map

model_key = "meta-llama/Llama-3.3-70B-Instruct"

device_map = get_device_map(model_key, 30, n_gpus=8) #! NOTE: CHANGE n_gpus=2 FOR VAST AI
device_map

2025-07-16 17:31:16 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/gio/mechinterp, stdin=None, shell=False, universal_newlines=False)
2025-07-16 17:31:16 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/gio/mechinterp, stdin=None, shell=False, universal_newlines=False)


{'model.embed_tokens': 7,
 'model.norm': 7,
 'model.rotary_emb': 7,
 'lm_head': 7,
 'model.layers.0': 0,
 'model.layers.1': 1,
 'model.layers.2': 2,
 'model.layers.3': 3,
 'model.layers.4': 4,
 'model.layers.5': 5,
 'model.layers.6': 6,
 'model.layers.7': 7,
 'model.layers.8': 0,
 'model.layers.9': 1,
 'model.layers.10': 2,
 'model.layers.11': 3,
 'model.layers.12': 4,
 'model.layers.13': 5,
 'model.layers.14': 6,
 'model.layers.15': 7,
 'model.layers.16': 0,
 'model.layers.17': 1,
 'model.layers.18': 2,
 'model.layers.19': 3,
 'model.layers.20': 4,
 'model.layers.21': 5,
 'model.layers.22': 6,
 'model.layers.23': 7,
 'model.layers.24': 0,
 'model.layers.25': 1,
 'model.layers.26': 2,
 'model.layers.27': 3,
 'model.layers.28': 4,
 'model.layers.29': 5,
 'model.layers.30': 0,
 'model.layers.31': 1,
 'model.layers.32': 2,
 'model.layers.33': 3,
 'model.layers.34': 4,
 'model.layers.35': 5,
 'model.layers.36': 6,
 'model.layers.37': 7,
 'model.layers.38': 0,
 'model.layers.39': 1,
 'model

In [7]:
os.getcwd()

'/disk/u/gio/mechinterp'

In [8]:
from src.models import ModelandTokenizer

mt = ModelandTokenizer(
    model_key=model_key,
    torch_dtype=torch.bfloat16,
    device_map=device_map,
    #device_map="balanced",
    #max_memory={i: f"{25 if i<7 else 21}GiB" for i in range(8)}
)

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

2025-07-16 17:32:31 src.models INFO     loaded model <models/meta-llama/Llama-3.3-70B-Instruct> | size: 134570.516 MB | dtype: torch.bfloat16 | device: cuda:7


In [9]:
os.getcwd()
os.chdir("../")
os.getcwd()

'/disk/u/gio'

In [10]:
from src.functional import free_gpu_cache

SYNTH_DATASET = "test_72"

#checkpoint_path = os.path.join(
#    "results",
#    "trained_params",
#    f"test_72",
#    "_full__clamp=0.001", 
#    model_key.split("/")[-1],
#    "epoch_1"
#)

#checkpoint_path = "results/trained_params/test_72/_full__clamp=0.001/Llama-3.3-70B-Instruct/epoch_1/"
checkpoint_path = "trained_models/_full__clamp=0.001/Llama-3.3-70B-Instruct/epoch_1/"

print(os.listdir(checkpoint_path))

checkpoint_path = os.path.join(checkpoint_path, "trainable_params.pt")

loaded_deltas = torch.load(checkpoint_path, map_location="cpu")

free_gpu_cache()

d = loaded_deltas['model<>layers<>10<>mlp<>gate_proj']
d.abs().max()

['trainable_params.pt']


tensor(0.0010, dtype=torch.bfloat16, grad_fn=<MaxBackward1>)

In [11]:
from src.utils.training_utils import TrainableLM_delta

Trainable_CLS = TrainableLM_delta

Trainable_CLS.fuse_with_model(mt._model, loaded_deltas)

2025-07-16 17:33:14 src.utils.training_utils DEBUG    module_name='model.layers.0.mlp.gate_proj' | param_delta.shape=torch.Size([28672, 8192])


2025-07-16 17:33:15 src.utils.training_utils DEBUG    module_name='model.layers.0.mlp.up_proj' | param_delta.shape=torch.Size([28672, 8192])
2025-07-16 17:33:15 src.utils.training_utils DEBUG    module_name='model.layers.0.mlp.down_proj' | param_delta.shape=torch.Size([8192, 28672])
2025-07-16 17:33:15 src.utils.training_utils DEBUG    module_name='model.layers.1.mlp.gate_proj' | param_delta.shape=torch.Size([28672, 8192])
2025-07-16 17:33:15 src.utils.training_utils DEBUG    module_name='model.layers.1.mlp.up_proj' | param_delta.shape=torch.Size([28672, 8192])
2025-07-16 17:33:15 src.utils.training_utils DEBUG    module_name='model.layers.1.mlp.down_proj' | param_delta.shape=torch.Size([8192, 28672])
2025-07-16 17:33:15 src.utils.training_utils DEBUG    module_name='model.layers.2.mlp.gate_proj' | param_delta.shape=torch.Size([28672, 8192])
2025-07-16 17:33:15 src.utils.training_utils DEBUG    module_name='model.layers.2.mlp.up_proj' | param_delta.shape=torch.Size([28672, 8192])
2025-

In [13]:
import os

os.getcwd()
os.chdir("./mechinterp")
os.getcwd()

'/disk/u/gio/mechinterp'

In [15]:
from src.dataset import ActivationPatchingChoiceSamples

target_attribute="profession"
result_sub_dir="act_patch_choose_one_common_middle"

activation_patching_path = os.path.join(
    "results",
    result_sub_dir,
    "test_72",
    target_attribute
)

samples_path = os.path.join(
    activation_patching_path, f"samples.json"
)

samples = []
with open(samples_path, "r") as f:
    samples = json.load(f)
samples = [ActivationPatchingChoiceSamples.from_dict(sample) for sample in samples]

In [16]:
samples

[ActivationPatchingChoiceSamples(prompt_template='Q:Which of the following people has a profession in common with {}? {}.\nA:', common_entities=['Somchai Jaidee', 'Grace Wanjiru', 'Marie Laurent'], clean_entity='Valentina Lopez', patched_entity='Fatima Sheikh', clean_answer=' None', patched_answer=' Grace', patched_answer_toks=[32171]),
 ActivationPatchingChoiceSamples(prompt_template='Q:Which of the following people has a profession in common with {}? {}.\nA:', common_entities=['Jose Cruz', 'Nguyen Van Duc', 'Erik Andersson'], clean_entity='Yuki Tanaka', patched_entity='Chinedu Okafor', clean_answer=' None', patched_answer=' Nguyen', patched_answer_toks=[64261]),
 ActivationPatchingChoiceSamples(prompt_template='Q:Which of the following people has a profession in common with {}? {}.\nA:', common_entities=['Jennifer Davis', 'Ayse Kaya', 'Jose Cruz'], clean_entity='Amara Adeyemi', patched_entity='Hans Mueller', clean_answer=' None', patched_answer=' Ay', patched_answer_toks=[24852]),
 A

In [14]:
from src.trace import CausalTracingResult, patched_run, get_score, calculate_indirect_effects
from src.tokens import prepare_input, insert_padding_before_subj, find_token_range
from src.functional import get_all_module_states, interpret_logits
from src.utils.typing import PredictedToken
from src.dataset import ActivationPatchingSamples

def align_patching_positions(
    mt: ModelandTokenizer,
    prompt_template: str,
    clean_subj: str,
    patched_subj: str,
    common_entities: list[str],
    clean_entities: list[str],
    corrupt_entities: list[str],
    clean_input = None,
    patched_input = None,
    trace_start_marker = None
):
    # If clean input isn't provided, prepare it with the prompt template,
    # clean subject, and the clean entities
    if clean_input is None:
        clean_input = prepare_input(
            prompts=prompt_template.format(clean_subj, (", ").join(clean_entities)),
            tokenizer=mt,
            return_offsets_mapping=True,
        )
    else:
        assert "offset_mapping" in clean_input

    # If patched input isn't provided, prepare it with the prompt template,
    # patched subject and the corrupt entities
    if patched_input is None:
        patched_input = prepare_input(
            prompts=prompt_template.format(patched_subj, (", ").join(corrupt_entities)),
            tokenizer=mt,
            return_offsets_mapping=True,
        )
    else:
        assert "offset_mapping" in patched_input

    clean_subj_range = find_token_range(
        string=prompt_template.format(clean_subj, (", ").join(clean_entities)),
        substring=clean_subj,
        tokenizer=mt.tokenizer,
        occurrence=-1,
    )
    print(f"{clean_subj_range=}")

    patched_subj_range = find_token_range(
        string=prompt_template.format(patched_subj, (", ").join(corrupt_entities)),
        substring=patched_subj,
        tokenizer=mt.tokenizer,
        occurrence=-1
    )
    print(f"{patched_subj_range=}")

    trace_start_idx = None
    if trace_start_marker is not None:
        trace_start_idx = (
            find_token_range(
                string=prompt_template.format(clean_subj, (", ").join(common_entities)),
                substring=trace_start_marker,
                tokenizer=mt.tokenizer,
                occurrence=-1,
                offset_mapping=clean_input["offset_mapping"][0],
            )[1]
            -1
        )
        print(f"{trace_start_idx=}")
        assert trace_start_idx <= min(
            clean_subj_range[0], patched_subj_range[0]
        ), f"{trace_start_idx=} has to be smaller than {min(clean_subj_range[0], patched_subj_range[0])=}"

    if clean_subj_range == patched_subj_range:
        subj_start, subj_end = clean_subj_range
    else:
        subj_end = max(clean_subj_range[1], patched_subj_range[1])
        clean_input = insert_padding_before_subj(
            inp=clean_input,
            subj_range=clean_subj_range,
            subj_ends=subj_end,
            pad_id=mt.tokenizer.pad_token_id,
            fill_attn_mask=True,
        )
        patched_input = insert_padding_before_subj(
            inp=patched_input,
            subj_range=patched_subj_range,
            subj_ends=subj_end,
            pad_id=mt.tokenizer.pad_token_id,
            fill_attn_mask=True,
        )

        clean_subj_shift = subj_end - clean_subj_range[1]
        clean_subj_range = (clean_subj_range[0] + clean_subj_shift, subj_end)
        patched_subj_shift = subj_end - patched_subj_range[1]
        patched_subj_range = (patched_subj_range[0] + patched_subj_shift, subj_end)
        subj_start = min(clean_subj_range[0], patched_subj_range[0])

        if trace_start_idx is not None:
            trace_start_idx += clean_subj_shift

    # NOTE: Maybe need to add logic for fixing different common entitiy positions
    # since Arnab recommended I use different entities in the different prompts

    return dict(
        clean_input=clean_input,
        patched_input=patched_input,
        subj_range=(subj_start, subj_end),
        trace_start_idx=trace_start_idx,
    )

@torch.inference_mode()
def trace_important_states(
    mt: ModelandTokenizer,
    prompt_template,
    clean_subj,
    patched_subj,
    common_entities,
    clean_entities,
    corrupt_entities,
    clean_input = None,
    patched_input = None,
    kind = 'residual',
    window_size = 1,
    normalize = True,
    trace_start_marker = None,
    metric = "logit",
    ans_tokens = None,
) -> CausalTracingResult:
    aligned = align_patching_positions(
        mt=mt,
        prompt_template=prompt_template,
        clean_subj=clean_subj,
        patched_subj=patched_subj,
        common_entities=common_entities,
        clean_entities=clean_entities,
        corrupt_entities=corrupt_entities,
        clean_input=clean_input,
        patched_input=patched_input,
        trace_start_marker=trace_start_marker,
    )

    clean_input = aligned["clean_input"]
    print(f"Decoded clean input: {mt.tokenizer.decode(clean_input['input_ids'][0])}")
    
    patched_input = aligned["patched_input"]
    print(f"Decoded patched input: {mt.tokenizer.decode(patched_input['input_ids'][0])}")

    subj_range = aligned["subj_range"]
    trace_start_idx = aligned["trace_start_idx"]

    print(f"===> {trace_start_idx=}")

    return

    
    

In [ ]:
for kind in ["residual", "attention", "mlp"]:
    trace_results = trace_important_states(
        mt=mt,
        prompt_template=sample.prompt_template,
    )